In [5]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

from utils.data_utils import get_file_lists
from config import  INTERNAL_DIR,  DATA_DIR, EXTERNAL_DIR

file_lists = get_file_lists(os.path.join(DATA_DIR, INTERNAL_DIR))
print("Danh sách file CSV đã tạo:", file_lists)

# Combine all features into a single DataFrame
all_features = []
for file in file_lists:
    df = pd.read_csv(os.path.join(DATA_DIR, INTERNAL_DIR, file))
    all_features.append(df)

df = pd.concat(all_features, ignore_index=True)
print(df.shape)
print(df.head())

Danh sách file CSV đã tạo: ['features_sleep-cassette_SC4001.csv', 'features_sleep-cassette_SC4002.csv', 'features_sleep-cassette_SC4011.csv', 'features_sleep-cassette_SC4012.csv', 'features_sleep-cassette_SC4021.csv', 'features_sleep-cassette_SC4022.csv', 'features_sleep-cassette_SC4031.csv', 'features_sleep-cassette_SC4032.csv', 'features_sleep-cassette_SC4041.csv', 'features_sleep-cassette_SC4042.csv', 'features_sleep-cassette_SC4051.csv', 'features_sleep-cassette_SC4052.csv', 'features_sleep-cassette_SC4061.csv', 'features_sleep-cassette_SC4062.csv', 'features_sleep-cassette_SC4071.csv', 'features_sleep-cassette_SC4072.csv', 'features_sleep-cassette_SC4081.csv', 'features_sleep-cassette_SC4082.csv', 'features_sleep-cassette_SC4091.csv', 'features_sleep-cassette_SC4092.csv', 'features_sleep-cassette_SC4101.csv', 'features_sleep-cassette_SC4102.csv', 'features_sleep-cassette_SC4111.csv', 'features_sleep-cassette_SC4112.csv', 'features_sleep-cassette_SC4121.csv', 'features_sleep-casset

In [2]:
# Preprocessing
X = df.drop(columns=['start_time', 'index', 'label'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Cross-validation
rf = RandomForestClassifier(n_estimators=100, random_state=42)
cv_scores = cross_val_score(rf, X_train_scaled, y_train, cv=5)
print("Cross-validation scores:", cv_scores)
print("Average CV score:", np.mean(cv_scores))

rf.fit(X_train_scaled, y_train)
y_pred = rf.predict(X_test_scaled)

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Cross-validation scores: [0.74766672 0.74862832 0.74478194 0.74957574 0.74719991]
Average CV score: 0.7475705274693518
Confusion Matrix:
 [[1276  367  172   20  101]
 [ 411 1092 1173    7  416]
 [ 124  310 9662  262  446]
 [  39    2  499 1618    2]
 [ 123  303  695   10 2969]]
Classification Report:
               precision    recall  f1-score   support

           0       0.65      0.66      0.65      1936
           1       0.53      0.35      0.42      3099
           2       0.79      0.89      0.84     10804
           3       0.84      0.75      0.79      2160
           4       0.75      0.72      0.74      4100

    accuracy                           0.75     22099
   macro avg       0.71      0.68      0.69     22099
weighted avg       0.74      0.75      0.74     22099



In [7]:
# Test in External Dataset
external_file_lists = get_file_lists(os.path.join(DATA_DIR, EXTERNAL_DIR))
external_features = []
for file in external_file_lists:
    df_ext = pd.read_csv(os.path.join(DATA_DIR, EXTERNAL_DIR, file))
    external_features.append(df_ext)

df_external = pd.concat(external_features, ignore_index=True)
print(df_external.shape)

X_external = df_external.drop(columns=['start_time', 'index', 'label'])
y_external = df_external['label']

X_external_scaled = scaler.transform(X_external)
y_external_pred = rf.predict(X_external_scaled)

print("External Dataset Classification Report:\n", classification_report(y_external, y_external_pred))

(40111, 11)
External Dataset Classification Report:
               precision    recall  f1-score   support

           0       0.21      0.82      0.34      2244
           1       0.20      0.08      0.12      3601
           2       0.81      0.75      0.78     19674
           3       0.78      0.58      0.67      6283
           4       0.71      0.59      0.65      8309

    accuracy                           0.64     40111
   macro avg       0.54      0.57      0.51     40111
weighted avg       0.69      0.64      0.65     40111



In [5]:
import joblib
joblib.dump(model, 'models/random_forest_model.joblib')

['models/random_forest_model.joblib']